# RZ-USFM project · Part 4 — Bayesian late fusion, conformal BI-RADS 4 triage and full comparison

**Inputs:** out-of-fold predictions of the three experts — A (radiomics, Part 1), B (deep dual-stream, Part 2), C (RZ-USFM, Part 3) — with their uncertainties and the radiomic consistency score $c$.

**Proposed fusion (uncertainty- and consistency-gated hierarchical Bayesian stacking)**

$$w_{ik}=\mathrm{softplus}\big(\alpha_{k,s_i}+\gamma_k\,\tilde u_{ik}+\delta_k\,\tilde c_i\big),\qquad
\eta_i=\beta_0+\sum_k w_{ik}\,\mu_{ik},\qquad v_i=\sum_k w_{ik}^2u_{ik}^2$$
$$p_i=\sigma\!\Big(\eta_i\big/\sqrt{1+\pi v_i/8}\Big),\qquad y_i\sim\mathrm{Bernoulli}(p_i)$$
$$\alpha_{k,s}=\bar\alpha_k+\tau_k z_{s,k},\; z\sim\mathcal N(0,1),\;\bar\alpha_k\sim\mathcal N(0,1),\;\tau_k\sim\mathcal{HN}(0.5),\;\gamma_k,\delta_k\sim\mathcal N(0,0.5^2),\;\beta_0\sim\mathcal N(\mathrm{logit}\,\hat\pi,1)$$

$\mu_{ik}$ = expert logit, $u_{ik}$ = expert uncertainty (ensemble / bootstrap SD), $\tilde u,\tilde c$ = standardised log-uncertainty and log-consistency, $s_i$ = scanner. Posterior by NUTS (PyMC); predictions average over the posterior.

**Compared models:** each expert alone · USFM-FT (λ = 0) · probability averaging · logistic-regression stacking · early fusion (feature concatenation) · ungated Bayesian stacking (γ = δ = 0, no scanner effect) · **proposed gated Bayesian fusion** · the same fusion with USFM-FT instead of RZ-USFM (ablation) · the fusion without the consistency score (ablation).

**Evaluation:** outer patient-level folds (the meta-learner of fold *f* is fitted on the other folds' out-of-fold predictions); all metrics with 95% CI, DeLong tests, ROC / PR / calibration / confusion matrices, decision-curve analysis, **conformal BI-RADS 4 triage with a guaranteed sensitivity**, posterior analysis of the expert weights, SHAP and LIME on the fusion.

## 1. Setup and paths

In [ ]:
!pip -q install pymc arviz shap lime
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, warnings
warnings.filterwarnings('ignore')

PART1_DIR = '/content/drive/MyDrive/rsusfm_project'
PART4_DIR = '/content/drive/MyDrive/rsusfm_project_part4'          # ALL outputs of this notebook
def find(cands, pattern):
    d = next((c for c in cands if glob.glob(f'{c}/{pattern}')), None)
    assert d, f'{pattern} not found in {cands}'; return d
P1_RES = f'{PART1_DIR}/results'
P2_RES = find(['/content/drive/MyDrive/rsusfm_project_part2/results', P1_RES], 'oof_expertB.csv')
P3_RES = find(['/content/drive/MyDrive/rsusfm_project_part3/results', P1_RES], 'oof_expertC.csv')
FIG_DIR, RES_DIR = f'{PART4_DIR}/figures', f'{PART4_DIR}/results'
for d in (FIG_DIR, RES_DIR): os.makedirs(d, exist_ok=True)
print('Part 1:', P1_RES, '\nPart 2:', P2_RES, '\nPart 3:', P3_RES, '\nPart 4 outputs:', PART4_DIR)

SEED = 42
DRAWS, TUNE, CHAINS = 1000, 1000, 2      # NUTS settings (about 1 min per fit)
ALPHA_MISS = 0.02                        # conformal: at most 2% of cancers may be classified 'biopsy avoidable'
ALPHA_FP   = 0.10                        # conformal: at most 10% of benign lesions may be classified 'malignant'

In [ ]:
sys.path.insert(0, PART1_DIR)
import importlib, common; importlib.reload(common)
from common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
master = pd.read_csv(f'{PART1_DIR}/master.csv')
y_all = master.label.values; FOLDS = sorted(master.fold.unique())

def load_oof(path, extra=None):
    o = pd.read_csv(path).set_index('id').loc[master.id].reset_index()
    if extra and os.path.exists(extra):
        o = o.merge(pd.read_csv(extra)[['id', 'consistency']], on='id', how='left')
    return o
OOF = {
    'Expert A (radiomics)':  load_oof(f'{P1_RES}/oof_expertA.csv'),
    'Expert B (deep)':       load_oof(f'{P2_RES}/oof_expertB.csv'),
    'Expert C (RZ-USFM)':    load_oof(f'{P3_RES}/oof_expertC.csv', f'{P3_RES}/expertC_extra.csv'),
    'USFM-FT (λ=0)':         load_oof(f'{P3_RES}/oof_USFM-FT.csv', f'{P3_RES}/USFM-FT_extra.csv'),
}
for n, o in OOF.items():
    assert (o.label.values == y_all).all() and o.prob.notna().all(), n
    print(f'{n:22s} AUC {roc_auc_score(y_all, o.prob):.3f} | uncertainty available: {o.logit_sd.notna().all()}')

birads = master['birads'].astype(str).str.strip().str.upper() if 'birads' in master else pd.Series(['NA'] * len(master))
BR4 = birads.str.startswith('4').values
dev = master['device'].astype(str).values if 'device' in master else np.array(['all'] * len(master))
DEV_LEVELS = sorted(np.unique(dev)); S_IDX = np.array([DEV_LEVELS.index(d) for d in dev])
print(f'BI-RADS 4 lesions: {BR4.sum()} ({y_all[BR4].sum()} malignant) | scanners: {DEV_LEVELS}')

## 2. Meta-features and fusion models
Every fusion model is fitted **inside each outer fold** on the other folds' out-of-fold expert outputs and applied to the held-out fold (cross-validated stacking). Standardisation of the log-uncertainties and log-consistency uses the training folds only.

In [ ]:
import pymc as pm, pytensor.tensor as pt, arviz as az
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def meta(experts):
    """experts: list of OOF frames (the last one carries the consistency score)."""
    mu = np.column_stack([o.logit_mean.values for o in experts])
    u = np.column_stack([o.logit_sd.fillna(0).values for o in experts])
    has_u = np.array([o.logit_sd.notna().all() for o in experts], float)
    c = np.log1p(experts[-1]['consistency'].values) if 'consistency' in experts[-1] else np.zeros(len(mu))
    return dict(mu=mu, u=u, has_u=has_u, c=c)

def standardise(M, tr):
    lu = np.log(M['u'] + 1e-3); m_, s_ = lu[tr].mean(0), lu[tr].std(0) + 1e-8
    cm, cs = M['c'][tr].mean(), M['c'][tr].std() + 1e-8
    return (lu - m_) / s_ * M['has_u'], (M['c'] - cm) / cs

def fit_bayes(M, tr, gated=True, scanner=True, use_c=True, seed=SEED):
    lu, cc = standardise(M, tr); K = M['mu'].shape[1]; S = len(DEV_LEVELS) if scanner else 1
    s = S_IDX if scanner else np.zeros(len(S_IDX), int)
    prev = y_all[tr].mean()
    with pm.Model():
        b0 = pm.Normal('b0', np.log(prev / (1 - prev)), 1)
        abar = pm.Normal('abar', 0, 1, shape=K)
        if scanner and S > 1:
            tau = pm.HalfNormal('tau', 0.5, shape=K); z = pm.Normal('z', 0, 1, shape=(S, K))
            alpha = abar + tau * z
        else:
            alpha = pt.tile(abar[None, :], (S, 1))
        lin = alpha[s[tr]]
        if gated:
            g = pm.Normal('gamma', 0, 0.5, shape=K); lin = lin + g * M['has_u'] * lu[tr]
            if use_c:
                d = pm.Normal('delta', 0, 0.5, shape=K); lin = lin + d * cc[tr][:, None]
        w = pt.softplus(lin)
        eta = b0 + (w * M['mu'][tr]).sum(1); v = (w ** 2 * M['u'][tr] ** 2).sum(1)
        pm.Bernoulli('y', p=pm.math.sigmoid(eta / pt.sqrt(1 + np.pi * v / 8)), observed=y_all[tr])
        idata = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, cores=CHAINS, target_accept=0.9,
                          progressbar=False, random_seed=seed)
    return dict(idata=idata, lu=lu, cc=cc, gated=gated, scanner=scanner and S > 1, use_c=use_c, K=K)

def PV(idata, var):
    # posterior draws of a variable as (n_draws, ...) - works with ArviZ InferenceData and DataTree
    v = np.asarray(idata.posterior[var].values); return v.reshape(-1, *v.shape[2:])

def max_rhat(idata, names=('b0', 'abar', 'gamma', 'delta', 'tau')):
    out = 1.0
    for n in names:
        try: v = np.asarray(idata.posterior[n].values)
        except Exception: continue
        c, d = v.shape[:2]; v = v.reshape(c, d, -1)
        h = d // 2; v = np.concatenate([v[:, :h], v[:, h:2 * h]], 0)        # split chains
        m = v.shape[1]; W = v.var(1, ddof=1).mean(0); B = m * v.mean(1).var(0, ddof=1)
        out = max(out, float(np.nanmax(np.sqrt(((m - 1) / m * W + B / m) / (W + 1e-12)))))
    return out

def predict_bayes(F, M, idx, n_draws=400, return_w=False):
    I = F['idata']; b0_all = PV(I, 'b0')
    sel = np.linspace(0, len(b0_all) - 1, n_draws).astype(int)
    b0 = b0_all[sel]; abar = PV(I, 'abar')[sel]                                            # (D,), (D, K)
    if F['scanner']:
        alpha = abar[:, None, :] + PV(I, 'tau')[sel][:, None, :] * PV(I, 'z')[sel]          # (D, S, K)
        A = alpha[:, S_IDX[idx], :]                                                         # (D, n, K)
    else:
        A = np.repeat(abar[:, None, :], len(idx), 1)
    lin = A.copy()
    if F['gated']:
        lin += PV(I, 'gamma')[sel][:, None, :] * (M['has_u'] * F['lu'][idx])[None]
        if F['use_c']: lin += PV(I, 'delta')[sel][:, None, :] * F['cc'][idx][None, :, None]
    w = np.logaddexp(0, lin)
    eta = b0[:, None] + (w * M['mu'][idx][None]).sum(-1); v = (w ** 2 * M['u'][idx][None] ** 2).sum(-1)
    p = 1 / (1 + np.exp(-eta / np.sqrt(1 + np.pi * v / 8)))
    out = (p.mean(0), np.percentile(p, 2.5, 0), np.percentile(p, 97.5, 0))
    return (*out, w.mean(0)) if return_w else out

def frame_from(prob, thr_fn, name):
    o = master[['id', 'patient', 'fold', 'label']].copy(); o['prob'] = prob
    o['logit_mean'] = logit(prob); o['logit_sd'] = np.nan; o['thr'] = thr_fn
    return o

## 3. Early-fusion baseline inputs
Selected radiomic features (the fold-specific mRMR set of Part 1) + PCA(32) of the Expert B and Expert C embeddings, concatenated and classified with L2 logistic regression (all transforms fitted on training folds).

In [ ]:
rad = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{PART1_DIR}/radiomics/rad_orig_part*.csv'))]).drop_duplicates('id')
rad = rad.set_index('id').loc[master.id]
sel_feats = pd.read_csv(f'{P1_RES}/expertA_selected_features.csv')
EMB_B = np.load(f'{P2_RES}/expertB_emb_oof.npy') if os.path.exists(f'{P2_RES}/expertB_emb_oof.npy') else None
EMB_C = np.load(f'{P3_RES}/expertC_emb_oof.npy') if os.path.exists(f'{P3_RES}/expertC_emb_oof.npy') else None
print('radiomics', rad.shape, '| emb B', None if EMB_B is None else EMB_B.shape, '| emb C', None if EMB_C is None else EMB_C.shape)

def early_fusion(f):
    tr, te = (master.fold != f).values, (master.fold == f).values
    cols = sel_feats[f'fold{f}'].dropna().tolist()
    Xr = rad[cols].values.astype(float); Xr = np.where(np.isnan(Xr), np.nanmedian(Xr[tr], 0), Xr)
    blocks_tr, blocks_te = [Xr[tr]], [Xr[te]]
    for E in (EMB_B, EMB_C):
        if E is None: continue
        E = np.nan_to_num(E.astype(float))
        pca = PCA(32, random_state=SEED).fit(StandardScaler().fit_transform(E[tr]))
        sc = StandardScaler().fit(E[tr])
        blocks_tr.append(pca.transform(sc.transform(E[tr]))); blocks_te.append(pca.transform(sc.transform(E[te])))
    Xtr, Xte = np.hstack(blocks_tr), np.hstack(blocks_te)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(C=0.05, class_weight='balanced', max_iter=5000).fit(sc.transform(Xtr), y_all[tr])
    return m.predict_proba(sc.transform(Xte))[:, 1], m.predict_proba(sc.transform(Xtr))[:, 1]

## 4. Run all fusion models fold by fold
About 4 Bayesian fits per fold (≈ 1 min each). Results are cached on Drive, so a re-run skips finished folds.

In [ ]:
A, B, Cc, FT = (OOF[k] for k in ('Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'USFM-FT (λ=0)'))
M_RZ, M_FT = meta([A, B, Cc]), meta([A, B, FT])
BAYES = {   # name: (meta-features, gated, scanner, use consistency)
    'Bayesian stacking (ungated)':           (M_RZ, False, False, False),
    'Proposed (gated Bayesian fusion)':       (M_RZ, True, True, True),
    'Ablation - no consistency score':      (M_RZ, True, True, False),
    'Ablation - USFM-FT instead of RZ-USFM':   (M_FT, True, True, True),
}
CACHE = f'{RES_DIR}/fusion_oof_cache.npz'
P = dict(np.load(CACHE, allow_pickle=True)) if os.path.exists(CACHE) else {}
T0 = time.time()
for f in FOLDS:
    if f'done_{f}' in P: continue
    tr, te = (master.fold != f).values, (master.fold == f).values
    te_idx, tr_idx = np.where(te)[0], np.where(tr)[0]
    res = {}
    res['Average (A+B+C)'] = (np.mean([o.prob.values for o in (A, B, Cc)], 0), None)
    lr = LogisticRegression(C=1.0, max_iter=5000).fit(M_RZ['mu'][tr], y_all[tr])
    res['LR stacking'] = (lr.predict_proba(M_RZ['mu'])[:, 1], None)
    pe, pe_tr = early_fusion(f); tmp = np.full(len(master), np.nan); tmp[te] = pe; tmp[tr] = pe_tr
    res['Early fusion (concatenation)'] = (tmp, None)
    for name, (M, gated, scanner, use_c) in BAYES.items():
        t = time.time(); F = fit_bayes(M, tr, gated, scanner, use_c)
        p_te, lo, hi = predict_bayes(F, M, te_idx); p_tr = predict_bayes(F, M, tr_idx)[0]
        tmp = np.full(len(master), np.nan); tmp[te] = p_te; tmp[tr] = p_tr
        lo_ = np.full(len(master), np.nan); hi_ = lo_.copy(); lo_[te] = lo; hi_[te] = hi
        res[name] = (tmp, (lo_, hi_))
        rhat = max_rhat(F['idata'])
        print(f'  fold {f} | {name:38s} AUC {roc_auc_score(y_all[te], p_te):.3f} | max R-hat {rhat:.3f} | {time.time() - t:.0f}s')
    for name, (p, ci) in res.items():
        thr = youden_threshold(y_all[tr], p[tr])                   # threshold from training folds only
        P[f'{name}|prob|{f}'] = p[te]; P[f'{name}|thr|{f}'] = thr
        if ci is not None: P[f'{name}|lo|{f}'] = ci[0][te]; P[f'{name}|hi|{f}'] = ci[1][te]
    P[f'done_{f}'] = True
    np.savez(CACHE, **P)
    print(f'>>> fold {f} done ({(time.time() - T0) / 60:.1f} min)')

FUSION_NAMES = ['Average (A+B+C)', 'LR stacking', 'Early fusion (concatenation)'] + list(BAYES)
FUSED = {}
for name in FUSION_NAMES:
    o = master[['id', 'patient', 'fold', 'label']].copy()
    o['prob'] = np.nan; o['thr'] = np.nan; o['lo'] = np.nan; o['hi'] = np.nan
    for f in FOLDS:
        te = (master.fold == f).values
        o.loc[te, 'prob'] = P[f'{name}|prob|{f}']; o.loc[te, 'thr'] = float(P[f'{name}|thr|{f}'])
        if f'{name}|lo|{f}' in P: o.loc[te, 'lo'] = P[f'{name}|lo|{f}']; o.loc[te, 'hi'] = P[f'{name}|hi|{f}']
    o['logit_mean'] = logit(o.prob); o['logit_sd'] = np.nan
    FUSED[name] = o
PROPOSED = 'Proposed (gated Bayesian fusion)'
save_oof(FUSED[PROPOSED], 'fusion_proposed', RES_DIR)
FUSED[PROPOSED][['id', 'prob', 'lo', 'hi']].to_csv(f'{RES_DIR}/fusion_proposed_credible_intervals.csv', index=False)

## 5. Full comparison: metrics (95% CI) and DeLong tests against the proposed fusion

In [ ]:
ALL = {**OOF, **FUSED}
rows = [evaluate_oof(o, n, RES_DIR) for n, o in ALL.items()]
table = summary_table(rows, metrics=('AUC', 'Accuracy', 'Balanced_Acc', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'Brier'))
display(table); table.to_csv(f'{RES_DIR}/final_comparison_table.csv')
pd.DataFrame(rows).to_csv(f'{RES_DIR}/final_comparison_full.csv', index=False)

def delong(y, p1, p2):
    y = np.asarray(y).astype(bool); Pm = np.vstack([p1, p2]); m, n = y.sum(), (~y).sum()
    def midrank(x):
        o = np.argsort(x); xs = x[o]; r = np.zeros(len(x)); i = 0
        while i < len(x):
            j = i
            while j < len(x) and xs[j] == xs[i]: j += 1
            r[i:j] = 0.5 * (i + j - 1) + 1; i = j
        out = np.empty(len(x)); out[o] = r; return out
    tx = np.array([midrank(a) for a in Pm[:, y]]); ty = np.array([midrank(a) for a in Pm[:, ~y]]); tz = np.array([midrank(a) for a in Pm])
    auc = tz[:, y].sum(1) / m / n - (m + 1) / (2 * n)
    v01 = (tz[:, y] - tx) / n; v10 = 1 - (tz[:, ~y] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    z = (auc[0] - auc[1]) / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1] + 1e-12)
    return auc[0], auc[1], 2 * stats.norm.sf(abs(z))

def brier_boot(y, p1, p2, groups, n=2000, seed=SEED):
    rng = np.random.default_rng(seed); ug = np.unique(groups); ix = {g: np.where(groups == g)[0] for g in ug}; d = []
    for _ in range(n):
        idx = np.concatenate([ix[g] for g in rng.choice(ug, len(ug))])
        d.append(np.mean((p1[idx] - y[idx]) ** 2) - np.mean((p2[idx] - y[idx]) ** 2))
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

g = master.patient.astype(str).values; pp = FUSED[PROPOSED].prob.values
tests = []
for n, o in ALL.items():
    if n == PROPOSED: continue
    a1, a2, p = delong(y_all, pp, o.prob.values); db = brier_boot(y_all, pp, o.prob.values, g)
    tests.append({'compared with': n, 'AUC proposed': a1, 'AUC other': a2, 'ΔAUC': a1 - a2, 'p (DeLong)': p,
                  'ΔBrier (proposed − other)': db[0], 'ΔBrier 95% CI': f'{db[1]:.4f} to {db[2]:.4f}'})
tests = pd.DataFrame(tests); display(tests.round(4)); tests.to_csv(f'{RES_DIR}/tests_vs_proposed.csv', index=False)

## 6. Figures: ROC, PR, calibration, confusion matrices, metric bars

In [ ]:
MAIN = {'Expert A (radiomics)': OOF['Expert A (radiomics)'], 'Expert B (deep)': OOF['Expert B (deep)'],
        'Expert C (RZ-USFM)': OOF['Expert C (RZ-USFM)'], 'LR stacking': FUSED['LR stacking'],
        'Early fusion (concatenation)': FUSED['Early fusion (concatenation)'], PROPOSED: FUSED[PROPOSED]}
plot_roc(MAIN, f'{FIG_DIR}/roc_final.png', 'Experts and fusion (out-of-fold)')
plot_pr(MAIN, f'{FIG_DIR}/pr_final.png')
plot_calibration(MAIN, f'{FIG_DIR}/calibration_final.png')
import re
safe = lambda n: re.sub(r'[^A-Za-z0-9_-]+', '_', n).strip('_')
for n in (PROPOSED, 'LR stacking', 'Early fusion (concatenation)'): plot_confusion(FUSED[n], n, f'{FIG_DIR}/cm_{safe(n)}.png')
plot_metric_bars([r for r in rows if r['model'] in MAIN], f'{FIG_DIR}/metrics_bars_final.png')
ABL = {k: FUSED[k] for k in BAYES}
plot_roc(ABL, f'{FIG_DIR}/roc_ablation_bayes.png', 'Bayesian fusion variants (ablation)')
plot_metric_bars([r for r in rows if r['model'] in ABL], f'{FIG_DIR}/metrics_bars_ablation.png')

## 7. Decision-curve analysis (net benefit of biopsy decisions)

In [ ]:
pts = np.linspace(0.02, 0.6, 59)
def net_benefit(y, p, t):
    tp = ((p >= t) & (y == 1)).sum(); fp = ((p >= t) & (y == 0)).sum(); n = len(y)
    return tp / n - fp / n * t / (1 - t)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for n, o in MAIN.items(): ax.plot(pts, [net_benefit(y_all, o.prob.values, t) for t in pts], lw=2 if n == PROPOSED else 1.2, label=n)
prev = y_all.mean(); ax.plot(pts, prev - (1 - prev) * pts / (1 - pts), 'k--', lw=1, label='Biopsy all'); ax.axhline(0, c='grey', lw=1, label='Biopsy none')
ax.set(xlabel='Threshold probability', ylabel='Net benefit', ylim=(-0.05, prev + 0.05), title='Decision-curve analysis')
ax.legend(fontsize=7); fig.savefig(f'{FIG_DIR}/decision_curve.png', bbox_inches='tight'); plt.show()

## 8. Conformal BI-RADS 4 triage with a guaranteed sensitivity
Class-conditional (Mondrian) split conformal, cross-fitted: for test fold *f*, the calibration set is the out-of-fold fused predictions of the other folds.
- $t_{\text{low}}$: the $\lfloor\alpha_1(n_1+1)\rfloor$-th smallest malignant calibration score → at most $\alpha_1$ of cancers fall below it.
- $t_{\text{high}}$: the same on benign lesions from the top, with $\alpha_0$.
- Decision: $p<t_{\text{low}}$ → **benign: biopsy avoidable**; $p>t_{\text{high}}$ → **malignant**; otherwise **uncertain → biopsy**.

Reported on the **BI-RADS 4** subgroup (all lesions in parentheses): achieved sensitivity (cancers *not* sent home) and the fraction of benign lesions whose biopsy is avoided.

In [ ]:
def conformal(prob, a1=ALPHA_MISS, a0=ALPHA_FP):
    n1 = min(((master.fold != f).values & (y_all == 1)).sum() for f in FOLDS)
    if np.floor(a1 * (n1 + 1)) < 1:
        print(f'note: α = {a1} needs at least {int(np.ceil(1 / a1 - 1))} malignant calibration lesions (have {n1}) -> no lesion can be sent home')
    dec = np.empty(len(prob), object)
    for f in FOLDS:
        cal, te = (master.fold != f).values, (master.fold == f).values
        s1 = np.sort(prob[cal & (y_all == 1)]); k1 = int(np.floor(a1 * (len(s1) + 1)))
        s0 = np.sort(prob[cal & (y_all == 0)])[::-1]; k0 = int(np.floor(a0 * (len(s0) + 1)))
        t_low = s1[k1 - 1] if k1 >= 1 else -np.inf; t_high = s0[k0 - 1] if k0 >= 1 else np.inf
        p = prob[te]
        dec[te] = np.where(p < t_low, 'benign (avoid biopsy)', np.where(p > t_high, 'malignant', 'uncertain (biopsy)'))
    return dec

def triage_summary(dec, mask):
    y = y_all[mask]; d = dec[mask]
    sent_home = d == 'benign (avoid biopsy)'
    return {'sensitivity (cancers not sent home)': 1 - (sent_home & (y == 1)).sum() / max(1, (y == 1).sum()),
            'benign biopsies avoided': (sent_home & (y == 0)).sum() / max(1, (y == 0).sum()),
            'n benign avoided': int((sent_home & (y == 0)).sum()), 'n cancers missed': int((sent_home & (y == 1)).sum()),
            'uncertain rate': (d == 'uncertain (biopsy)').mean()}

TRI = {**MAIN, 'Ablation - USFM-FT instead of RZ-USFM': FUSED['Ablation - USFM-FT instead of RZ-USFM'],
       'Bayesian stacking (ungated)': FUSED['Bayesian stacking (ungated)']}
tri_rows = []
for n, o in TRI.items():
    dec = conformal(o.prob.values)
    for label, mask in (('BI-RADS 4', BR4), ('all lesions', np.ones(len(master), bool))):
        if mask.sum() == 0: continue
        tri_rows.append({'model': n, 'subset': label, **triage_summary(dec, mask)})
    if n == PROPOSED:
        master_dec = dec
tri = pd.DataFrame(tri_rows); display(tri.round(3)); tri.to_csv(f'{RES_DIR}/conformal_triage.csv', index=False)
pd.DataFrame({'id': master.id, 'birads': birads, 'label': y_all, 'decision': master_dec}).to_csv(f'{RES_DIR}/triage_decisions_proposed.csv', index=False)
display(pd.crosstab(pd.Series(master_dec[BR4], name='decision (BI-RADS 4)'), pd.Series(np.where(y_all[BR4] == 1, 'malignant', 'benign'), name='pathology')))

alphas = [0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10]
mask = BR4 if BR4.sum() > 0 else np.ones(len(master), bool)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for n, o in TRI.items():
    s = [triage_summary(conformal(o.prob.values, a1=a), mask) for a in alphas]
    axes[0].plot([100 * a for a in alphas], [100 * x['benign biopsies avoided'] for x in s], 'o-', lw=2 if n == PROPOSED else 1, label=n)
    axes[1].plot([100 * a for a in alphas], [100 * x['sensitivity (cancers not sent home)'] for x in s], 'o-', lw=2 if n == PROPOSED else 1)
axes[1].plot([0, 10], [100, 90], 'k--', lw=1, label='guarantee 1 − α')
axes[0].set(xlabel='Allowed miss rate α (%)', ylabel='Benign biopsies avoided (%)', title=f'Biopsy avoidance ({"BI-RADS 4" if BR4.sum() else "all"})')
axes[1].set(xlabel='Allowed miss rate α (%)', ylabel='Achieved sensitivity (%)', title='Achieved vs guaranteed sensitivity'); axes[1].legend(fontsize=8)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(f'{FIG_DIR}/conformal_triage_curves.png', bbox_inches='tight'); plt.show()

## 9. What did the fusion learn? Posterior of the expert weights (model fitted on all out-of-fold data, for interpretation only)
- $\mathrm{softplus}(\bar\alpha_k)$: average trust in each expert (94% central credible intervals in the table).
- $\gamma_k$: change of trust when the expert is more uncertain (expected < 0).
- $\delta_k$: change of trust when RZ-USFM's radiomic view is inconsistent with the measured radiomics.
- Per-scanner trust and per-lesion weight distributions.

In [ ]:
ALL_IDX = np.arange(len(master))
F_all = fit_bayes(M_RZ, np.ones(len(master), bool), gated=True, scanner=True, use_c=True)
EXP = ['A: radiomics', 'B: deep', 'C: RZ-USFM']
srows = []
for var in ('b0', 'abar', 'gamma', 'delta', 'tau'):
    try: v = PV(F_all['idata'], var)
    except Exception: continue
    v = v.reshape(len(v), -1)
    for k in range(v.shape[1]):
        lo, hi = np.percentile(v[:, k], [3, 97])
        srows.append({'parameter': var if v.shape[1] == 1 else f'{var} [{EXP[k]}]', 'mean': v[:, k].mean(), 'sd': v[:, k].std(),
                      'HDI 3%': lo, 'HDI 97%': hi, 'P(>0)': (v[:, k] > 0).mean()})
summ = pd.DataFrame(srows).set_index('parameter')
print('max R-hat:', round(max_rhat(F_all['idata']), 3))
display(summ.round(3)); summ.to_csv(f'{RES_DIR}/fusion_posterior_summary.csv')

class _Post:                                            # thin accessor used by the plots below
    def __getitem__(self, k): return type('V', (), {'values': PV(F_all['idata'], k)})()
post = _Post()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (var, title, tf) in zip(axes, [('abar', 'Average expert weight softplus(ᾱ)', lambda x: np.logaddexp(0, x)),
                                       ('gamma', 'Uncertainty effect γ', lambda x: x), ('delta', 'Consistency effect δ', lambda x: x)]):
    v = tf(post[var].values.reshape(-1, len(EXP)))
    ax.violinplot([v[:, k] for k in range(len(EXP))], showmeans=True)
    ax.set_xticks(range(1, len(EXP) + 1), EXP, fontsize=8); ax.set_title(title, fontsize=10)
    if var != 'abar': ax.axhline(0, c='k', lw=.8, ls='--')
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fusion_posterior_weights.png', bbox_inches='tight'); plt.show()

_, _, _, W = predict_bayes(F_all, M_RZ, ALL_IDX, return_w=True)
share = W / W.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].violinplot([share[:, k] for k in range(len(EXP))], showmeans=True); axes[0].set_xticks(range(1, len(EXP) + 1), EXP)
axes[0].set(ylabel='Share of total weight', title='Per-lesion expert weights')
if len(DEV_LEVELS) > 1:
    dfw = pd.DataFrame(share, columns=EXP); dfw['scanner'] = dev
    dfw.groupby('scanner').mean().plot.bar(ax=axes[1], rot=0); axes[1].set(ylabel='Mean weight share', title='Expert trust by scanner')
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fusion_lesion_weights.png', bbox_inches='tight'); plt.show()

## 10. Explainability of the fusion — SHAP and LIME
Inputs of the fusion: the three expert logits, their (log) uncertainties and the consistency score. SHAP (permutation explainer) on the posterior-mean predictor; LIME on four representative lesions (TP, TN, FP, FN of the proposed model), plus each expert's contribution $w_{ik}\mu_{ik}$ to the fused logit.

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
lu_all, cc_all = standardise(M_RZ, np.ones(len(master), bool))
FEAT = ['logit A (radiomics)', 'logit B (deep)', 'logit C (RZ-USFM)', 'log-unc. A', 'log-unc. B', 'log-unc. C', 'consistency c']
X = np.column_stack([M_RZ['mu'], lu_all, cc_all])
mu_u = M_RZ['u']
def fused_from_X(Xq, s_idx=None):
    """posterior-mean fused probability for arbitrary inputs (uncertainty SD rebuilt from the log-unc. features)."""
    pm_ = {k: PV(F_all['idata'], k).mean(0) for k in ('b0', 'abar', 'gamma', 'delta')}
    lin = pm_['abar'][None] + pm_['gamma'][None] * M_RZ['has_u'] * Xq[:, 3:6] + pm_['delta'][None] * Xq[:, 6:7]
    w = np.logaddexp(0, lin); u = np.exp(Xq[:, 3:6] * np.log(mu_u + 1e-3).std(0) + np.log(mu_u + 1e-3).mean(0)) * M_RZ['has_u']
    eta = pm_['b0'] + (w * Xq[:, :3]).sum(1); v = (w ** 2 * u ** 2).sum(1)
    return 1 / (1 + np.exp(-eta / np.sqrt(1 + np.pi * v / 8)))
rng = np.random.default_rng(SEED)
bg = X[rng.choice(len(X), 100, replace=False)]; xs = X[rng.choice(len(X), min(400, len(X)), replace=False)]
sv = shap.PermutationExplainer(fused_from_X, shap.maskers.Independent(bg), feature_names=FEAT)(xs)
plt.figure(); shap.plots.beeswarm(sv, max_display=len(FEAT), show=False); plt.title('SHAP — proposed Bayesian fusion')
plt.savefig(f'{FIG_DIR}/shap_fusion_beeswarm.png', bbox_inches='tight'); plt.show()
plt.figure(); shap.plots.bar(sv, show=False); plt.savefig(f'{FIG_DIR}/shap_fusion_bar.png', bbox_inches='tight'); plt.show()

o = FUSED[PROPOSED]; pred = (o.prob >= o.thr).astype(int).values
picks = {}
for name, cond in {'TP': (y_all == 1) & (pred == 1), 'TN': (y_all == 0) & (pred == 0),
                   'FP': (y_all == 0) & (pred == 1), 'FN': (y_all == 1) & (pred == 0)}.items():
    idx = np.where(cond)[0]
    if len(idx): picks[name] = idx[np.argmax(np.abs(o.logit_mean.values[idx]))]
lime_exp = LimeTabularExplainer(X, feature_names=FEAT, class_names=['benign', 'malignant'], discretize_continuous=True, random_state=SEED)
fig, axes = plt.subplots(2, len(picks), figsize=(4.2 * len(picks), 7.5)); axes = np.atleast_2d(axes.T).T if len(picks) == 1 else axes
for j, (name, i) in enumerate(picks.items()):
    e = lime_exp.explain_instance(X[i], lambda Z: np.column_stack([1 - fused_from_X(Z), fused_from_X(Z)]), num_features=6)
    wl = e.as_list()[::-1]
    axes[0, j].barh([a for a, _ in wl], [b for _, b in wl], color=['#C44E52' if b > 0 else '#4C72B0' for _, b in wl])
    axes[0, j].set_title(f'{name}: {["benign", "malignant"][y_all[i]]}, p = {o.prob.values[i]:.2f}', fontsize=9); axes[0, j].tick_params(labelsize=7)
    contrib = W[i] * M_RZ['mu'][i]
    axes[1, j].bar(EXP, contrib, color=['#55A868', '#8172B2', '#C44E52']); axes[1, j].axhline(0, c='k', lw=.8)
    axes[1, j].set_title('Expert contributions w·μ to the fused logit', fontsize=8); axes[1, j].tick_params(axis='x', labelsize=7)
fig.suptitle('LIME (top; red → malignant) and expert contributions (bottom) — proposed fusion'); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/lime_fusion_cases.png', bbox_inches='tight'); plt.show()

## 11. Outputs of Part 4 (all in `rsusfm_project_part4/`)
| File | Content |
|---|---|
| `results/final_comparison_table.csv`, `final_comparison_full.csv` | every model, all metrics with 95% CI (paper Table) |
| `results/tests_vs_proposed.csv` | DeLong ΔAUC and bootstrap ΔBrier vs the proposed fusion |
| `results/conformal_triage.csv`, `triage_decisions_proposed.csv` | BI-RADS 4 triage: guaranteed sensitivity, biopsies avoided |
| `results/fusion_posterior_summary.csv` | posterior means and 94% HDIs of the fusion parameters |
| `results/oof_fusion_proposed.csv`, `fusion_proposed_credible_intervals.csv` | fused probabilities with 95% credible intervals |
| `figures/*.png` | ROC, PR, calibration, confusion matrices, decision curves, triage curves, posterior weights, SHAP, LIME (300 dpi) |